In [ ]:
import re
import pandas as pd

# --- Archivos ---
IN_FILE  = "coppel.csv"         # Debe tener al menos la columna: Nombre
OUT_FILE = "coppel homologados.csv"

# ============== CONFIGURACIÓN =================
# Este script puede trabajar de dos formas:
# 1. Si el CSV tiene columna "Marca": usa esa información y la normaliza
# 2. Si no tiene columna "Marca": detecta la marca desde el nombre del producto
# 
# La detección automática desde el nombre sigue funcionando como respaldo
# para filas donde la columna "Marca" esté vacía o sea inválida.

# ============== DETECCIÓN DESDE "Nombre" =================

# 1) Patrones → MARCA (orden importa: primero los más específicos)
BRAND_PATTERNS = [
    (r"\bIPHONE\b|\bAPPLE\b", "APPLE"),
    (r"\bGALAXY Z FOLD\b|\bZ\s*FOLD\b", "SAMSUNG"),
    (r"\bGALAXY Z FLIP\b|\bZ\s*FLIP\b", "SAMSUNG"),
    (r"\bGALAXY\b|\bSAMSUNG\b|\bS\d{2,3}\b|\bA\d{2,3}\b|\bM\d{2,3}\b", "SAMSUNG"),
    (r"\bPIXEL\b|\bGOOGLE\b", "GOOGLE"),
    (r"\bONE\s*PLUS\b|\bONEPLUS\b", "ONEPLUS"),
    (r"\bMOTOROLA\b|\bMOTO\b", "MOTOROLA"),
    (r"\bPOCO\b", "XIAOMI"),
    (r"\bREDMI\b", "XIAOMI"),
    (r"\bXIAOMI\b|\bMI\b(?!\s*BAND)", "XIAOMI"),
    (r"\bHUAWEI\b", "HUAWEI"),
    (r"\bHONOR\b", "HONOR"),
    (r"\bOPPO\b", "OPPO"),
    (r"\bREALME\b", "REALME"),
    (r"\bVIVO\b", "VIVO"),
    (r"\bNOKIA\b", "NOKIA"),
    (r"\bZTE\b|\bBLADE\b", "ZTE"),
    (r"\bSONY\b|\bXPERIA\b", "SONY"),
    (r"\bLG\b", "LG"),
    (r"\bASUS\b|\bROG\b", "ASUS"),
    (r"\bTECNO\b", "TECNO"),
    (r"\bINFINIX\b", "INFINIX"),
    (r"\BITEL\b", "ITEL"),
    (r"\bLENOVO\b", "LENOVO"),
    (r"\bHISENSE\b", "HISENSE"),
    (r"\bTCL\b", "TCL"),
    (r"\bCUBOT\b", "CUBOT"),
]

# 2) Nombre comercial por marca (si aplica)
COMMERCIAL_BY_BRAND = {
    "APPLE": "iPhone",
    "SAMSUNG": "Galaxy",
    "MOTOROLA": "Motorola",
    "GOOGLE": "Pixel",
    "XIAOMI": "Redmi",
    "HUAWEI": "HUAWEI",
    "HONOR": "HONOR",
    "OPPO": "OPPO",
    "REALME": "realme",
    "VIVO": "vivo",
    "NOKIA": "Nokia",
    "ZTE": "ZTE",
    "SONY": "Xperia",
    "LG": "LG",
    "ONEPLUS": "OnePlus",
    "ASUS": "ROG",
    "TECNO": "TECNO",
    "INFINIX": "Infinix",
    "ITEL": "itel",
    "LENOVO": "Lenovo",
    "SOYES": "SOYES",
    "HISENSE": "Hisense",
    "TCL": "TCL",
    "CUBOT": "CUBOT",
}

# 3) Limpieza de ruido común en "Nombre"
NOISE_REGEXES = [
    r"\bCELULAR\b",
    r"\bLIBRE\b|\bLIBERADO\b",
    r"\bCOLOR\s+[A-ZÁÉÍÓÚÑ]+(?:\s+[A-ZÁÉÍÓÚÑ]+)?\b",   # "Color Verde"/"Color Azul Negro"
    r"\bR\d+\b",                                       # "R9"
    r"\(TELCEL\)|\(AT&T\)|\(MOVISTAR\)|\(UNEFON\)",
    r"[\[\(].*?[\]\)]",                                # cualquier paréntesis/llaves con contenido
    r"\bDS\b|\bDUAL\s*SIM\b",
    r"\b4\s*G\b|\b5\s*G\b",                            # 4G/5G
    r"\bAMOLED\b",                                     # AMOLED
    r"\bLCD\b",                                        # LCD
    r"\bOLED\b",                                       # OLED
    r"\b\d+\.?\d*\s*PULGADAS?\b",                      # "6.8 PULGADAS"
    r"\b\d+\.?\d*\s*INCHES?\b",                        # "6.8 INCHES"
    r"\b\d+\.?\d*\"\b",                                # 6.8"
    r"\bPULGADAS?\b",                                  # PULGADAS
    r"\bINCHES?\b",                                    # INCHES
    r"\bPANTALLA\b",                                   # PANTALLA
    r"\bSCREEN\b",                                     # SCREEN
    r"\bDISPLAY\b",                                    # DISPLAY
]

CAP_REGEX = re.compile(r"(\d+)\s*([gG][bB])")          # 128GB, 256 gb, etc.

def norm_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def upper_clean(s: str) -> str:
    s = str(s).upper()
    s = s.replace("·", " ")
    s = norm_spaces(s)
    return s

def detect_brand(nombre_up: str) -> str:
    for pattern, brand in BRAND_PATTERNS:
        if re.search(pattern, nombre_up, flags=re.I):
            return brand
    return "DESCONOCIDA"

def strip_noise(nombre_up: str) -> str:
    t = nombre_up
    for rx in NOISE_REGEXES:
        t = re.sub(rx, " ", t, flags=re.I)
    return norm_spaces(t)

def extract_capacity(nombre_up: str) -> str:
    last = None
    for m in CAP_REGEX.finditer(nombre_up):
        last = m
    return f"{last.group(1)}GB" if last else ""

def extract_model(nombre_up: str, brand: str) -> str:
    """
    Quita la marca y deja el modelo más 'útil'.
    Incluye patrones conocidos (iPhone 14 Pro Max, Galaxy S23 Ultra, Honor X7B, etc.)
    Para ZTE: agrega "BLADE" automáticamente si no está presente en el modelo.
    """
    t = strip_noise(nombre_up)

    # quitar keywords de marca del inicio o donde aparezcan
    brand_keywords = {
        "APPLE": [r"\bAPPLE\b", r"\bIPHONE\b"],
        "SAMSUNG": [r"\bSAMSUNG\b", r"\bGALAXY\b"],
        "MOTOROLA": [r"\bMOTOROLA\b", r"\bMOTO\b"],
        "BLU": [r"\bBLU\b"],
        "GOOGLE": [r"\bGOOGLE\b", r"\bPIXEL\b"],
        "XIAOMI": [r"\bXIAOMI\b", r"\bREDMI\b", r"\bPOCO\b", r"\bMI\b(?!\s*BAND)"],
        "HUAWEI": [r"\bHUAWEI\b"],
        "HONOR": [r"\bHONOR\b"],
        "OPPO": [r"\bOPPO\b"],
        "REALME": [r"\bREALME\b"],
        "VIVO": [r"\bVIVO\b"],
        "NOKIA": [r"\bNOKIA\b"],
        "ZTE": [r"\bZTE\b", r"\bBLADE\b"],
        "SONY": [r"\bSONY\b", r"\bXPERIA\b"],
        "LG": [r"\bLG\b"],
        "ONEPLUS": [r"\bONE\s*PLUS\b", r"\bONEPLUS\b"],
        "ASUS": [r"\bASUS\b", r"\bROG\b"],
        "TECNO": [r"\bTECNO\b"],
        "INFINIX": [r"\bINFINIX\b"],
        "ITEL": [r"\bITEL\b"],
        "LENOVO": [r"\bLENOVO\b"],
        "SOYES": [r"\bSOYES\b"],
        "HISENSE": [r"\bHISENSE\b"],
        "TCL": [r"\bTCL\b"],
        "ZUUM": [r"\bZUUM\b"],
        "CUBOT": [r"\bCUBOT\b"],
        "BLU": [r"\bBLU\b"],
        "DESCONOCIDA": []
    }

    for kw in brand_keywords.get(brand, []):
        t = re.sub(kw, " ", t, flags=re.I)

    t = norm_spaces(t)

    # Separar palabras pegadas con números y letras
    t = re.sub(r"\b(Z)(FLIP|FOLD)(\d+)\b", r"\1 \2 \3", t, flags=re.I)  # Z FLIP/FOLD
    t = re.sub(r"\b(RENO)(\d+)([A-Z]+)\b", r"\1 \2\3", t, flags=re.I)  # RENO13F → RENO 13F
    t = re.sub(r"\b(RENO)(\d+)\b", r"\1 \2", t, flags=re.I)  # RENO13 → RENO 13
    t = re.sub(r"\b(EDGE)(\d+)\b", r"\1 \2", t, flags=re.I)  # EDGE60 → EDGE 60
    t = re.sub(r"\b(BLADE)(\d+)\b", r"\1 \2", t, flags=re.I)  # BLADE10 → BLADE 10
    t = re.sub(r"\b(G)\s+(\d+)\b", r"\1\2", t, flags=re.I)  # G 24 → G24

    # cortar antes de una capacidad si quedó
    t = re.split(r"\b\d+\s*[gG][bB]\b", t)[0].strip()

    # patrones de modelos "compuestos" frecuentes (orden importa: más específicos primero)
    patterns = [
        r"IPHONE\s*\d{1,2}\s*(PRO\s*MAX|PRO|PLUS|MINI)?",
        r"Z\s+FLIP\s+\d{1,2}",
        r"Z\s+FOLD\s+\d{1,2}",
        r"MAGIC\s*\d+[A-Z]*",  # HONOR Magic (antes que patrones genéricos)
        r"VIEW\s*\d+[A-Z]*",   # HONOR View (antes que patrones genéricos)
        r"X\d+[A-Z]*",  # HONOR X7B, X9A, etc. (antes que patrones genéricos)
        r"S\d{2,3}\s*(ULTRA|PLUS)?",
        r"A\d{2,3}[A-Z]?",
        r"M\d{2,3}",
        r"EDGE\s*\d{1,3}\s*(FUSION|NEO|PRO|PLUS|ULTRA)?",
        r"G\d{1,3}",
        r"BLADE\s*[A-Z]?\d{0,3}\s*(V\d{1,3}\s*MAX?)?",
        r"AXON\s*\d{1,3}",  # ZTE AXON
        r"PIXEL\s*[A-Z]?\d{1,2}\s*(PRO|A)?",
        r"REDMI\s*\d{1,3}\s*[A-Z]?",
        r"POCO\s*[A-Z]?\d{1,3}\s*(PRO|X|F\d{1,2})?",
        r"XPERIA\s*[A-Z]?\d{1,2}",
        r"RENO\s*\d{1,3}[A-Z]*\s*(PRO|PLUS)?",
        r"REALME\s*\d{1,3}\s*[A-Z]?",
        r"V\d{2,3}\s*MAX?",
        r"NOKIA\s*\d{1,3}",
        r"Y\d{2,3}",
        r"C\d+",  # Modelos Xiaomi como "C75"
        r"\d+[A-Z]+",  # Patrón genérico (al final)
    ]

    for p in patterns:
        m = re.search(p, t, flags=re.I)
        if m:
            model = norm_spaces(m.group(0).upper())
            
            # Especial para ZTE: agregar "BLADE" si no está presente
            if brand == "ZTE" and not re.search(r"\bBLADE\b", model, flags=re.I):
                # Si el modelo no contiene "BLADE" y no es "AXON", agregar "BLADE"
                if not re.search(r"\bAXON\b", model, flags=re.I):
                    model = f"BLADE {model}"
            
            return model

    # si no hubo match, usa lo que quedó (limitado)
    remaining_model = norm_spaces(t.upper())
    
    # Especial para ZTE: agregar "BLADE" si no está presente
    if brand == "ZTE" and remaining_model and not re.search(r"\bBLADE\b", remaining_model, flags=re.I):
        # Si el modelo no contiene "BLADE" y no es "AXON", agregar "BLADE"
        if not re.search(r"\bAXON\b", remaining_model, flags=re.I):
            remaining_model = f"BLADE {remaining_model}"
    
    return remaining_model

def commercial_name(brand: str) -> str:
    return COMMERCIAL_BY_BRAND.get(brand, brand.title())

def normalize_brand_from_csv(brand_str: str) -> str:
    """
    Normaliza las marcas que vienen del CSV para que coincidan con nuestro formato estándar.
    """
    if not brand_str or str(brand_str).upper() in ["", "NAN", "NONE", "NULL"]:
        return ""
    
    brand_clean = upper_clean(brand_str)
    
    # Mapeo de variaciones comunes a nuestras marcas estándar
    brand_mappings = {
        "APPLE": "APPLE",
        "IPHONE": "APPLE",
        "SAMSUNG": "SAMSUNG",
        "GALAXY": "SAMSUNG",
        "XIAOMI": "XIAOMI", 
        "REDMI": "XIAOMI",
        "POCO": "XIAOMI",
        "MI": "XIAOMI",
        "HUAWEI": "HUAWEI",
        "HONOR": "HONOR",
        "MOTOROLA": "MOTOROLA",
        "MOTO": "MOTOROLA",
        "GOOGLE": "GOOGLE",
        "PIXEL": "GOOGLE",
        "ONEPLUS": "ONEPLUS",
        "ONE PLUS": "ONEPLUS",
        "OPPO": "OPPO",
        "REALME": "REALME",
        "VIVO": "VIVO",
        "NOKIA": "NOKIA",
        "ZTE": "ZTE",
        "BLADE": "ZTE",
        "SONY": "SONY",
        "XPERIA": "SONY",
        "LG": "LG",
        "ASUS": "ASUS",
        "ROG": "ASUS",
        "TECNO": "TECNO",
        "INFINIX": "INFINIX",
        "ITEL": "ITEL",
        "LENOVO": "LENOVO",
        "HISENSE": "HISENSE",
        "TCL": "TCL",
        "CUBOT": "CUBOT",
    }
    
    # Buscar coincidencia exacta primero
    if brand_clean in brand_mappings:
        return brand_mappings[brand_clean]
    
    # Buscar coincidencia parcial (para casos como "SAMSUNG GALAXY" -> "SAMSUNG")
    for key, value in brand_mappings.items():
        if key in brand_clean or brand_clean in key:
            return value
    
    # Si no encuentra coincidencia, devolver la marca limpia
    return brand_clean

def build_homologated(row):
    """
    Construye el nombre homologado final limpiando colores, carriers y especificaciones técnicas.
    """
    parts = [row["MARCA"]]
    
    modelo = row["MODELO"]
    
    # Limpiar colores, carriers y otros elementos no deseados del modelo
    if modelo:
        # Patrones de colores comunes en español e inglés
        color_patterns = [
            r"\b(NEGRO|BLACK|AZULADO|DEL DESIERTO|BLANCO|WHITE|AZUL|BLUE|ROJO|RED|VERDE|GREEN|AMARILLO|YELLOW|ROSA|PINK|DORADO|GOLD|PLATEADO|SILVER|GRIS|GRAY|GREY|MORADO|PURPLE|VIOLETA|VIOLET|NARANJA|ORANGE|CAFE|BROWN|TITANIO|TITANIUM|NATURAL|MIDNIGHT|CLARO|STARLIGHT|CORAL|SIERRA|PACIFIC|ALPINE|COSMIC|GRAPHITE|PHANTOM|MYSTIC|LAVENDER|CREAM|SAGE|SORBET|MINT|CORAL|SUNSET|AURORA|OCEAN|FROST|STORM|CIELO|TITANIO\s+NEGRO|TITANIO\s+BLANCO|TITANIO\s+DEL\s+DESIERTO|TITANIO\s+NATURAL|PURPURA|CIAN|LILA|GRAFITO|PURPURA|LAVANDA|ULTRAMARINO|MEDIANOCHE|MENTA|MULTICOLOR|BEIGE|VIOLETA)\b",
            r"\bCOLOR\s+\w+\b"
        ]
        
        # Patrones de carriers
        carrier_patterns = [
            r"\b(TELCEL|AT&T|MOVISTAR|UNEFON|BAIT|VIRGIN|PILLOFON|WEEX|CIERTO|FLASH|LIBRE|LIBERADO|UNLOCKED)\b",
            r"\(TELCEL\)|\(AT&T\)|\(MOVISTAR\)|\(UNEFON\)"
        ]
        
        # Patrones de palabras adicionales a eliminar
        additional_patterns = [
            r"\bAUTORIZADO\b", r"\bCERTIFICADO\b", r"\bDISTRIBUIDOR\b", r"\bBUNDLE\b",
            r"\bREACONDICIONADO\s*GRADO\s*[A-Z]\b", r"\bREACONDICIONADO\b", r"\bGRADO\s*[A-Z]\b",
            r"\bUSADO\b", r"\bREFURBISHED\b", r"\bOPEN\s*BOX\b", r"\bDEMO\b", r"\bEXHIBICION\b",
            r"\bMUESTRA\b", r"\bRENOVADO\b", r"\bSMARTPHONE\b", r"\bRESTAURADO\b", r"\bDE\b",
            r"\bWATCH\b", r"\bSMARTWATCH\b", r"\bESIM\b", r"\bSELLADO\b", r"\bCARGADOR\b",
            r"\bINALAMBRICO\b", r"\bMAGNETICO\b", r"\bNUEVO\b", r"\bBOSSNEY\b", r"\bAUDÃFONOS\b",
            r"\bAUDIFONOS\b", r"\bTIDE\b", r"\bBS\b", r"\bK\d+\b", r"\bAC\d+\b", r"\b\d+/\d+\b",
            r"\bCON\b", r"\bBLUETOOTH\b", r"\bBUDS\b", r"\bY\b", r"\bFILWANS\b",
            r"\bAMOLED\b", r"\bLCD\b", r"\bOLED\b", r"\bPULGADAS?\b", r"\bINCHES?\b",
            r"\b\d+\.?\d*\s*PULGADAS?\b", r"\b\d+\.?\d*\s*INCHES?\b", r"\b\d+\.?\d*\"\b",
            r"\bPANTALLA\b", r"\bSCREEN\b", r"\bDISPLAY\b",
            r"[,;]", r"\s*-\s*", r"\s*\+\s*"
        ]
        
        # Aplicar todas las limpiezas
        for pattern in color_patterns + carrier_patterns + additional_patterns:
            modelo = re.sub(pattern, " ", modelo, flags=re.I)
        
        modelo = norm_spaces(modelo)
    
    # Para Apple, agregar "IPHONE" si no está ya en el modelo
    if row["MARCA"] == "APPLE" and modelo:
        if not modelo.upper().startswith("IPHONE"):
            parts.append("IPHONE")
        parts.append(modelo)
    elif modelo:
        parts.append(modelo)
    
    if row["CAPACIDAD"]:
        # Eliminar espacios en la capacidad para que sea 256GB en lugar de 256 GB
        capacidad_sin_espacios = row["CAPACIDAD"].replace(" ", "")
        parts.append(capacidad_sin_espacios)
    
    return norm_spaces(" ".join(parts))

# ================== PROCESAMIENTO ==================

df = pd.read_csv(IN_FILE, encoding='latin-1')

# normaliza cabeceras
df.rename(columns={c: c.strip().title() for c in df.columns}, inplace=True)
if "Nombre" not in df.columns:
    raise ValueError("El CSV debe tener una columna 'Nombre'.")

df["NOMBRE_UP"] = df["Nombre"].apply(upper_clean)

# Detectar MARCA: primero desde columna "Marca" si existe, sino desde "Nombre"
if "Marca" in df.columns:
    print("✅ Columna 'Marca' encontrada. Usando información de marca del CSV.")
    # Usar la marca del CSV, limpiar y normalizar
    df["MARCA"] = df["Marca"].fillna("").astype(str).apply(normalize_brand_from_csv)
    
    # Si hay valores vacíos en la columna Marca, intentar detectar desde el nombre
    mask_empty_marca = (df["MARCA"] == "") | (df["MARCA"] == "NAN") | (df["MARCA"] == "NONE")
    if mask_empty_marca.any():
        print(f"⚠️ {mask_empty_marca.sum()} filas con marca vacía. Intentando detectar desde el nombre...")
        df.loc[mask_empty_marca, "MARCA"] = df.loc[mask_empty_marca, "NOMBRE_UP"].apply(detect_brand)
else:
    print("⚠️ Columna 'Marca' no encontrada. Detectando marca desde el nombre del producto.")
    # Detectar MARCA desde "Nombre" como antes
    df["MARCA"] = df["NOMBRE_UP"].apply(detect_brand)

# Extraer MODELO desde "Nombre"
df["MODELO"] = df.apply(lambda r: extract_model(r["NOMBRE_UP"], r["MARCA"]), axis=1)

# Usar capacidad del CSV si existe, sino extraer del nombre
if "Capacidad" in df.columns:
    # Usar la capacidad del CSV original
    df["CAPACIDAD"] = df["Capacidad"].fillna("").astype(str)
    # Limpiar valores vacíos o nulos
    df["CAPACIDAD"] = df["CAPACIDAD"].replace(["nan", "NaN", "None", ""], "")
else:
    # Si no existe la columna, extraer del nombre
    df["CAPACIDAD"] = df["NOMBRE_UP"].apply(extract_capacity)

# Nombre comercial por marca
df["NOMBRE_COMERCIAL"] = df["MARCA"].apply(commercial_name)

# Mostrar estadísticas de marcas detectadas
print(f"\n📊 Estadísticas de marcas detectadas:")
marca_counts = df["MARCA"].value_counts()
print(marca_counts.to_string())

if "DESCONOCIDA" in marca_counts.index:
    unknown_count = marca_counts["DESCONOCIDA"]
    total_count = len(df)
    print(f"\n⚠️ {unknown_count} de {total_count} productos ({unknown_count/total_count*100:.1f}%) tienen marca DESCONOCIDA")
    
    # Mostrar algunos ejemplos de productos con marca desconocida
    unknown_examples = df[df["MARCA"] == "DESCONOCIDA"]["Nombre"].head(5).tolist()
    print("Ejemplos de productos con marca desconocida:")
    for i, example in enumerate(unknown_examples, 1):
        print(f"  {i}. {example}")
        
print(f"\n✅ Total de marcas únicas detectadas: {len(marca_counts)}")

# Construir nombre homologado
df["NOMBRE_HOMOLOGADO"] = df.apply(build_homologated, axis=1)

# Reporte de filas inciertas
missing_model = df.index[df["MODELO"].str.len() == 0].tolist()
missing_brand = df.index[df["MARCA"] == "DESCONOCIDA"].tolist()

if missing_brand or missing_model:
    print("⚠️ Revisión sugerida:")
    if missing_brand:
        print(f" - No se pudo detectar MARCA en filas: {missing_brand[:20]}{' ...' if len(missing_brand)>20 else ''}")
    if missing_model:
        print(f" - No se pudo detectar MODELO en filas: {missing_model[:20]}{' ...' if len(missing_model)>20 else ''}")

# Guardar salida
cols_out = ["Nombre", "MARCA", "MODELO", "NOMBRE_COMERCIAL", "CAPACIDAD", "NOMBRE_HOMOLOGADO"]
df[cols_out].to_csv(OUT_FILE, index=False)
print(f"✅ Archivo generado: {OUT_FILE}")

✅ Columna 'Marca' encontrada. Usando información de marca del CSV.

📊 Estadísticas de marcas detectadas:
MARCA
SAMSUNG     12
XIAOMI       6
OPPO         4
HONOR        4
REALME       3
NUBIA        3
VIVO         3
CUBOT        2
APPLE        2
INFINIX      1
ITEL         1
MOTOROLA     1

✅ Total de marcas únicas detectadas: 12
✅ Archivo generado: coppel homologados.csv


: 